In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


file = "TS hour raw value 2024 L1&2.xlsx"
df = pd.read_excel(file)

df.head()


# ============================================================
# SABİTLER
# ============================================================

CPA = 1.3104
CPF_TFUM = 1.087
CPF_TREF = 1.225
NCV_GAS = 33500.0
TREF = 15.0
VWA = 5.56
VO2_MCR = 0.0764
COLD_AIR_INJECTION = 5200.0 / 3600.0
H_EA = 551.0
H_WH = 3216.0
LOSS = 0.04


# ============================================================
# YARDIMCI FONKSİYONLAR
# ============================================================

def nm3h_to_nm3s(x):
    """Nm3/h -> Nm3/s"""
    return x / 3600.0


def th_to_kgs(x):
    """ton/h -> kg/s"""
    return x * 1000.0 / 3600.0


def pct_to_frac(x):
    """
    Yüzde değerlerini fraction'a çevirir.

    Örnek:
    8.5 -> 0.085
    0.085 -> 0.085
    """
    return np.where(x > 1, x / 100.0, x)


def clamp_nonnegative(x):
    """Negatif değerleri sıfıra çeker."""
    return np.maximum(x, 0)


# ============================================================
# KOLON EŞLEŞTİRME
# ============================================================

def get_cols(prefix):

    base = {
        "W_GR": f"{prefix}HJG15_CF001_XQ02||OUT",
        "W_GL": f"{prefix}HJG25_CF001_XQ02||OUT",
        "W_AS": f"{prefix}HLA10_CF001_XQ02||OUT",
        "T_AS": f"{prefix}HLA10_CT001_XQ02||OUT",
        "W_AP": f"{prefix}HLA20_CF001_XQ02||OUT",
        "T_AP": f"{prefix}HLA20_CT007_XQ02||OUT",
        "VO2": f"{prefix}HNA10_CQ011_XQ02||OUT",
        "TFUM": f"{prefix}HNA10_CT008|528160|XQ02",
        "VH2O": f"{prefix}HNY12_CQ906_ZQ02||OUT",
        "W_WH": f"{prefix}LBA30_CF001_XQ02||OUT",
    }

    # Waste Flow kolon isimleri L1 ve L2 için farklı
    if prefix == "1_":
        base["W_OM"] = "1_WASTE_FLOW||OUT"
    elif prefix == "2_":
        base["W_OM"] = "2_WASTE_FLOW||"
    else:
        raise ValueError(f"Bilinmeyen line prefix: {prefix}")

    return base


# ============================================================
# KOLON KONTROLÜ
# ============================================================

def validate_columns(df, cols, line_name):

    missing = [
        column
        for column in cols.values()
        if column not in df.columns
    ]

    if missing:
        print(f"\n{line_name} için eksik kolonlar:")

        for column in missing:
            print(f"  - {column}")

        raise KeyError(
            f"{line_name} hesabı yapılamadı. "
            f"{len(missing)} adet gerekli kolon Excel dosyasında bulunamadı."
        )


# ============================================================
# ANA HESAPLAMA
# ============================================================

def compute_line(df, prefix):

    cols = get_cols(prefix)

    line_name = "L1" if prefix == "1_" else "L2"

    # Önce kolonları kontrol et
    validate_columns(df, cols, line_name)

    # Orijinal dataframe'i değiştirmemek için copy
    data = df.copy()

    # Kullanılacak kolonları numeric yap
    for column in cols.values():
        data[column] = pd.to_numeric(
            data[column],
            errors="coerce"
        )

    # --------------------------------------------------------
    # FLOW CONVERSIONS
    # --------------------------------------------------------

    W_AP = nm3h_to_nm3s(data[cols["W_AP"]])
    W_AS = nm3h_to_nm3s(data[cols["W_AS"]])

    W_GR = nm3h_to_nm3s(data[cols["W_GR"]])
    W_GL = nm3h_to_nm3s(data[cols["W_GL"]])

    W_WH = th_to_kgs(data[cols["W_WH"]])
    W_OM = th_to_kgs(data[cols["W_OM"]])

    # --------------------------------------------------------
    # TEMPERATURES
    # --------------------------------------------------------

    T_AP = data[cols["T_AP"]]
    T_AS = data[cols["T_AS"]]
    T_FUM = data[cols["TFUM"]]

    # --------------------------------------------------------
    # FLUE GAS ANALYSIS
    # --------------------------------------------------------

    VO2 = pct_to_frac(data[cols["VO2"]])
    VH2O = pct_to_frac(data[cols["VH2O"]])

    # --------------------------------------------------------
    # FLUE GAS FLOW
    # --------------------------------------------------------

    denominator = 1 - VH2O

    # VH2O = 100% olması gibi matematiksel hataları önle
    denominator = np.where(
        np.abs(denominator) < 1e-9,
        np.nan,
        denominator
    )

    VWG = (
        VWA
        * (1 + VO2 - VO2_MCR)
        / denominator
    )

    WFUM = W_OM * VWG

    # --------------------------------------------------------
    # FLUE GAS HEAT
    # --------------------------------------------------------

    PFUM = WFUM * (
        CPF_TFUM * T_FUM
        - CPF_TREF * TREF
    )

    # --------------------------------------------------------
    # AIR HEAT
    # --------------------------------------------------------

    QAIR = CPA * (
        clamp_nonnegative(
            W_AP - COLD_AIR_INJECTION
        )
        * (T_AP - TREF)
        +
        W_AS * (T_AS - TREF)
    )

    # --------------------------------------------------------
    # WATER / STEAM ENERGY
    # --------------------------------------------------------

    QNR = W_WH * (
        H_WH - H_EA
    )

    # --------------------------------------------------------
    # GAS ENERGY
    # --------------------------------------------------------

    QGAS = NCV_GAS * (
        W_GR + W_GL
    )

    # --------------------------------------------------------
    # WASTE ENERGY
    # --------------------------------------------------------

    QOM = (
        QNR
        + PFUM
        - QAIR
        - QGAS
    ) / (1 - LOSS)

    # --------------------------------------------------------
    # NCV
    # --------------------------------------------------------

    NCV = QOM / W_OM.replace(
        0,
        np.nan
    )

    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------

    result = pd.DataFrame({
        "NCV": NCV,
        "QOM": QOM
    })

    return result


# ============================================================
# MAIN
# ============================================================

def main():

    # Bu Python dosyasının bulunduğu klasör
    BASE_DIR = Path.cwd()

    # Excel input
    input_file = (
        BASE_DIR
        / "data"
        / "TS hour raw value 2024 L1&2.xlsx"
    )

    # Output klasörü
    output_dir = BASE_DIR / "output"

    # Yoksa oluştur
    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # Output Excel
    output_file = (
        output_dir
        / "NCV_2024_hourly_L1_L2.xlsx"
    )

    # --------------------------------------------------------
    # INPUT FILE CONTROL
    # --------------------------------------------------------

    if not input_file.exists():

        raise FileNotFoundError(
            "\nInput Excel dosyası bulunamadı.\n"
            f"Beklenen konum:\n{input_file}\n\n"
            "Excel dosyasını repository içerisindeki "
            "'data' klasörüne koy."
        )

    print("=" * 60)
    print("NCV CALCULATION")
    print("=" * 60)

    print(f"\nInput:\n{input_file}")

    # --------------------------------------------------------
    # READ EXCEL
    # --------------------------------------------------------

    print("\nExcel okunuyor...")

    df = pd.read_excel(
        input_file,
        engine="openpyxl"
    )

    print(
        f"Excel okundu: "
        f"{len(df):,} satır x "
        f"{len(df.columns):,} kolon"
    )

    # --------------------------------------------------------
    # CALCULATIONS
    # --------------------------------------------------------

    print("\nL1 hesaplanıyor...")
    L1 = compute_line(df, "1_")

    print("L2 hesaplanıyor...")
    L2 = compute_line(df, "2_")

    # --------------------------------------------------------
    # SAVE EXCEL
    # --------------------------------------------------------

    print("\nSonuç Excel'e yazılıyor...")

    with pd.ExcelWriter(
        output_file,
        engine="openpyxl"
    ) as writer:

        L1.to_excel(
            writer,
            index=False,
            sheet_name="L1"
        )

        L2.to_excel(
            writer,
            index=False,
            sheet_name="L2"
        )

    print("\n" + "=" * 60)
    print("EXCEL OLUŞTURULDU")
    print("=" * 60)

    print(output_file)

    # --------------------------------------------------------
    # QUICK SUMMARY
    # --------------------------------------------------------

    print("\nL1 NCV Summary:")
    print(L1["NCV"].describe())

    print("\nL2 NCV Summary:")
    print(L2["NCV"].describe())


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'TS hour raw value 2024 L1&2.xlsx'

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# SABITLER
# ============================================================

CPA = 1.3104
CPF_TFUM = 1.087
CPF_TREF = 1.225
NCV_GAS = 33500.0
TREF = 15.0
VWA = 5.56
VO2_MCR = 0.0764
COLD_AIR_INJECTION = 5200.0 / 3600.0
H_EA = 551.0
H_WH = 3216.0
LOSS = 0.04


# ============================================================
# YARDIMCI FONKSIYONLAR
# ============================================================

def nm3h_to_nm3s(x):
    return x / 3600.0


def th_to_kgs(x):
    return x * 1000.0 / 3600.0


def pct_to_frac(x):
    return np.where(x > 1, x / 100.0, x)


def clamp_nonnegative(x):
    return np.maximum(x, 0)


# ============================================================
# KOLON ESLESTIRME
# ============================================================

def get_cols(prefix):

    base = {
        "W_GR": f"{prefix}HJG15_CF001_XQ02||OUT",
        "W_GL": f"{prefix}HJG25_CF001_XQ02||OUT",
        "W_AS": f"{prefix}HLA10_CF001_XQ02||OUT",
        "T_AS": f"{prefix}HLA10_CT001_XQ02||OUT",
        "W_AP": f"{prefix}HLA20_CF001_XQ02||OUT",
        "T_AP": f"{prefix}HLA20_CT007_XQ02||OUT",
        "VO2":  f"{prefix}HNA10_CQ011_XQ02||OUT",
        "TFUM": f"{prefix}HNA10_CT008|528160|XQ02",
        "VH2O": f"{prefix}HNY12_CQ906_ZQ02||OUT",
        "W_WH": f"{prefix}LBA30_CF001_XQ02||OUT",
    }

    if prefix == "1_":
        base["W_OM"] = "1_WASTE_FLOW||OUT"

    elif prefix == "2_":
        base["W_OM"] = "2_WASTE_FLOW||"

    else:
        raise ValueError(f"Gecersiz prefix: {prefix}")

    return base


# ============================================================
# KOLON KONTROLU
# ============================================================

def validate_columns(df, cols, line_name):

    missing = [
        col
        for col in cols.values()
        if col not in df.columns
    ]

    if missing:

        print(f"\n{line_name} icin eksik kolonlar:")

        for col in missing:
            print(" -", col)

        raise KeyError(
            f"{line_name} hesabi yapilamadi. "
            f"{len(missing)} kolon eksik."
        )


# ============================================================
# ANA HESAPLAMA
# ============================================================

def compute_line(df, prefix):

    cols = get_cols(prefix)

    line_name = "L1" if prefix == "1_" else "L2"

    validate_columns(df, cols, line_name)

    data = df.copy()

    # Gerekli kolonlari sayisal formata cevir
    for col in cols.values():
        data[col] = pd.to_numeric(
            data[col],
            errors="coerce"
        )

    # --------------------------------------------------------
    # FLOW
    # --------------------------------------------------------

    W_AP = nm3h_to_nm3s(data[cols["W_AP"]])
    W_AS = nm3h_to_nm3s(data[cols["W_AS"]])

    W_GR = nm3h_to_nm3s(data[cols["W_GR"]])
    W_GL = nm3h_to_nm3s(data[cols["W_GL"]])

    W_WH = th_to_kgs(data[cols["W_WH"]])
    W_OM = th_to_kgs(data[cols["W_OM"]])

    # --------------------------------------------------------
    # TEMPERATURE
    # --------------------------------------------------------

    T_AP = data[cols["T_AP"]]
    T_AS = data[cols["T_AS"]]
    T_FUM = data[cols["TFUM"]]

    # --------------------------------------------------------
    # GAS ANALYSIS
    # --------------------------------------------------------

    VO2 = pct_to_frac(data[cols["VO2"]])
    VH2O = pct_to_frac(data[cols["VH2O"]])

    # --------------------------------------------------------
    # FLUE GAS
    # --------------------------------------------------------

    denominator = 1 - VH2O

    denominator = np.where(
        np.abs(denominator) < 1e-9,
        np.nan,
        denominator
    )

    VWG = (
        VWA
        * (1 + VO2 - VO2_MCR)
        / denominator
    )

    WFUM = W_OM * VWG

    # --------------------------------------------------------
    # PFUM
    # --------------------------------------------------------

    PFUM = WFUM * (
        CPF_TFUM * T_FUM
        - CPF_TREF * TREF
    )

    # --------------------------------------------------------
    # QAIR
    # --------------------------------------------------------

    QAIR = CPA * (
        clamp_nonnegative(
            W_AP - COLD_AIR_INJECTION
        )
        * (T_AP - TREF)
        +
        W_AS * (T_AS - TREF)
    )

    # --------------------------------------------------------
    # QNR
    # --------------------------------------------------------

    QNR = W_WH * (
        H_WH - H_EA
    )

    # --------------------------------------------------------
    # QGAS
    # --------------------------------------------------------

    QGAS = NCV_GAS * (
        W_GR + W_GL
    )

    # --------------------------------------------------------
    # QOM
    # --------------------------------------------------------

    QOM = (
        QNR
        + PFUM
        - QAIR
        - QGAS
    ) / (1 - LOSS)

    # --------------------------------------------------------
    # NCV
    # --------------------------------------------------------

    W_OM_safe = W_OM.replace(
        0,
        np.nan
    )

    NCV = QOM / W_OM_safe

    # --------------------------------------------------------
    # RESULT
    # --------------------------------------------------------

    result = pd.DataFrame({
        "NCV": NCV,
        "QOM": QOM
    })

    return result


# ============================================================
# JUPYTER / CODESPACES AYARLARI
# ============================================================

# Notebook'un calistigi repository klasoru
BASE_DIR = Path.cwd()

# Input Excel
input_file = (
    BASE_DIR
    / "data"
    / "TS hour raw value 2024 L1&2.xlsx"
)

# Output klasoru
output_dir = BASE_DIR / "output"

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Output Excel
output_file = (
    output_dir
    / "NCV_2024_hourly_L1_L2.xlsx"
)


# ============================================================
# DOSYA KONTROLU
# ============================================================

print("Calisma klasoru:")
print(BASE_DIR)

print("\nInput dosyasi:")
print(input_file)

print("\nDosya var mi?:", input_file.exists())


if not input_file.exists():

    raise FileNotFoundError(
        "\nExcel dosyasi bulunamadi.\n"
        f"Beklenen yer:\n{input_file}\n\n"
        "Excel dosyasini 'data' klasorune koy."
    )


# ============================================================
# EXCEL OKU
# ============================================================

print("\nExcel okunuyor...")

df = pd.read_excel(
    input_file,
    engine="openpyxl"
)

print(
    f"Excel okundu: "
    f"{len(df):,} satir x "
    f"{len(df.columns):,} kolon"
)


# ============================================================
# HESAPLAMA
# ============================================================

print("\nL1 hesaplaniyor...")

L1 = compute_line(
    df,
    "1_"
)


print("L2 hesaplaniyor...")

L2 = compute_line(
    df,
    "2_"
)


# ============================================================
# EXCEL'E KAYDET
# ============================================================

print("\nSonuc Excel'e yaziliyor...")

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    L1.to_excel(
        writer,
        index=False,
        sheet_name="L1"
    )

    L2.to_excel(
        writer,
        index=False,
        sheet_name="L2"
    )


# ============================================================
# SONUC
# ============================================================

print("\n" + "=" * 60)
print("EXCEL OLUSTURULDU")
print("=" * 60)

print(output_file)


print("\nL1 NCV OZET:")
print(L1["NCV"].describe())


print("\nL2 NCV OZET:")
print(L2["NCV"].describe())


# İlk birkaç sonucu göster
print("\nL1 ilk 5 sonuc:")
display(L1.head())

print("\nL2 ilk 5 sonuc:")
display(L2.head())

Calisma klasoru:
/workspaces/codespaces-jupyter

Input dosyasi:
/workspaces/codespaces-jupyter/data/TS hour raw value 2024 L1&2.xlsx

Dosya var mi?: True

Excel okunuyor...


Excel okundu: 8,786 satir x 31 kolon

L1 hesaplaniyor...
L2 hesaplaniyor...

Sonuc Excel'e yaziliyor...

EXCEL OLUSTURULDU
/workspaces/codespaces-jupyter/output/NCV_2024_hourly_L1_L2.xlsx

L1 NCV OZET:
count    7.255000e+03
mean     6.352186e+03
std      5.605623e+04
min     -4.622311e+06
25%      6.003715e+03
50%      6.736557e+03
75%      7.335058e+03
max      7.602992e+05
Name: NCV, dtype: float64

L2 NCV OZET:
count     7310.000000
mean      6797.563875
std       1397.380574
min     -13588.385354
25%       6181.667054
50%       6840.100764
75%       7431.839461
max      32031.949137
Name: NCV, dtype: float64

L1 ilk 5 sonuc:


,NCV,QOM
0,NaN,NaN
1,5646.602363,58172.944464
2,7453.777822,72042.315516
3,8289.722699,80744.777465
4,8020.740360,77764.976765



L2 ilk 5 sonuc:


,NCV,QOM
0,NaN,NaN
1,6249.506277,63940.695095
2,6719.161571,69705.235385
3,8281.962612,81355.904252
4,7231.896745,73204.372588
